In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [ ]:
print('---LOAD CLEANED DATA ---')

file_path = "../data/processed/antimicrobial_clean.csv"
print("Loading dataset...\n")
df = pd.read_csv(file_path)

print(f"Total Rows: {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}\n")
df.info()
print()
display(df.head())

---LOAD CLEANED DATA ---
Loading dataset...

Total Rows: 211
Total Columns: 15

<class 'pandas.DataFrame'>
RangeIndex: 211 entries, 0 to 210
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   DOI                   211 non-null    str    
 1   YEAR                  211 non-null    int64  
 2   MATERIAL              211 non-null    str    
 3   PRECURSOR             211 non-null    str    
 4   Synthesis_Method      211 non-null    str    
 5   Size_nm               211 non-null    float64
 6   Plant Extract Type    211 non-null    str    
 7   Bacterial Strain      211 non-null    str    
 8   Inhibition Zone (mm)  129 non-null    float64
 9   MIC(ug/ml)            211 non-null    float64
 10  MBC                   211 non-null    float64
 11  ROS Generation        211 non-null    str    
 12  Gram Status           211 non-null    str    
 13  Resistant Type        211 non-null    str    
 14  IZ_Ce

,DOI,YEAR,MATERIAL,PRECURSOR,Synthesis_Method,Size_nm,Plant Extract Type,Bacterial Strain,Inhibition Zone (mm),MIC(ug/ml),MBC,ROS Generation,Gram Status,Resistant Type,IZ_Censored
0,10.1016/j.carbon.2021.10.020,2021,Carbon NPs,Carbon source,Hydrothermal,7.0,Unknown,S. aureus,20.0,0.125,130.0,Yes,Gram-positive,Unknown,Unknown
1,10.1016/j.carbon.2021.10.020,2021,Carbon NPs,Carbon source,Hydrothermal,7.0,Unknown,Other,20.0,0.500,130.0,Yes,Gram-positive,Unknown,Unknown
2,10.1016/j.carbon.2021.10.020,2021,Carbon NPs,Carbon source,Hydrothermal,7.0,Unknown,Other,20.0,0.500,130.0,Yes,Gram-positive,Unknown,Unknown
3,10.1016/j.carbon.2021.10.020,2021,Carbon NPs,Carbon source,Hydrothermal,7.0,Unknown,Other,20.0,0.500,130.0,Yes,Gram-positive,Unknown,Unknown
4,10.1016/j.carbon.2021.10.020,2021,Carbon NPs,Carbon source,Hydrothermal,7.0,Unknown,Other,20.0,0.250,130.0,Yes,Gram-positive,MRSA,Unknown


In [ ]:
# Drop rows where Inhibition Zone is null
df_model = df.dropna(subset=['Inhibition Zone (mm)']).copy()
print(f"Rows after dropping null IZ: {df_model.shape[0]}")

Rows after dropping null IZ: 129


In [2]:
print('---Define Target Variable(y) And Features(X) ---')
#Target Variable
y = df_model['Inhibition Zone (mm)']
X = df_model.drop(columns=['DOI','Inhibition Zone (mm)','YEAR'] ,errors='ignore')
print(f"\nFeature columns (X): {list(X.columns)}")
print(f"Target variable (y): Inhibition Zone (mm), shape: {y.shape}")



---Define Target Variable(y) And Features(X) ---

Feature columns (X): ['MATERIAL', 'PRECURSOR', 'Synthesis_Method', 'Size_nm', 'Plant Extract Type', 'More about Plant Extract Type', 'Bacterial Strain', 'MIC(ug/ml)', 'MBC', 'ROS Generation', 'Gram Status', 'Resistant Strain', 'Resistant Type', 'IZ_Censored']
Target variable (y): Inhibition Zone (mm), shape: (131,)


In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

url = "https://docs.google.com/spreadsheets/d/106rVYbCbHIljj3CKQtQnBp5rDmGtZA4ecyAEgQ277Dg/export?format=csv&gid=796927918"

df = pd.read_csv(url)
df.columns = df.columns.str.strip()
df_model = df.dropna(subset=['Inhibition Zone (mm)'])

doi_counts = df_model['DOI'].groupby(df_model['DOI']).transform('count')
df_small = df_model[doi_counts <= 4]
df_large = df_model[doi_counts > 4]

groups = pd.factorize(df_small['DOI'])[0]

gss = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)

train_pos, test_pos = next(gss.split(df_small, groups=groups))

test_indices = df_small.iloc[test_pos].index
train_indices = df_small.iloc[train_pos].index.union(df_large.index)


X_train = X.loc[train_indices]
X_test = X.loc[test_indices]
y_train = y.loc[train_indices]
y_test = y.loc[test_indices]

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_cols = ['MATERIAL', 'PRECURSOR', 'Synthesis_Method', 'Plant Extract Type',
                    'Bacterial Strain', 'ROS Generation', 'Gram Status'] # include all your cat cols

preprocessor = ColumnTransformer(
    transformers=[('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)],
    remainder='passthrough'
)

X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

print(f"X_train shape: {X_train_encoded.shape}")
print(f"X_test shape:  {X_test_encoded.shape}")


X_train shape: (106, 51)
X_test shape:  (25, 51)


In [8]:
import os

os.makedirs('data/splits', exist_ok=True)

pd.DataFrame(X_train_encoded).to_csv('data/splits/X_train.csv', index=False)
pd.DataFrame(X_test_encoded).to_csv('data/splits/X_test.csv', index=False)
y_train.to_csv('data/splits/y_train.csv', index=False)
y_test.to_csv('data/splits/y_test.csv', index=False)

print("Splits successfully saved to data/splits/!")

Splits successfully saved to data/splits/!
